# Практика 12. Статистический анализ и проверка гипотез

На практике 9 мы увидели, что отток по городам различается на 9.6 процентных пункта,
и сказали: «возможно, это случайность, проверять будем позже». На практике 10 сравнили
OTD перевозчиков — 70.1% против 77.9% — и снова отложили вопрос.

**Сегодня — тот самый «позже».** Учимся отвечать на вопрос, который висел два занятия:
*разница, которую я вижу в данных, настоящая или случайная?*

| Часть | О чём |
|---|---|
| 1 | Нулевая и альтернативная гипотезы |
| 2 | Что такое p-value (и чем оно не является) |
| 3 | Сравнение долей: критерий хи-квадрат |
| 4 | Сравнение средних: t-тест и Манна-Уитни |
| 5 | Связь двух категорий: таблица сопряжённости |
| 6 | Ловушки: размер выборки и множественные сравнения |
| 7 | Отчёт о проверке гипотез |

> **Главная мысль пары:** статистический тест не говорит «верно» или «неверно».
> Он отвечает только на один вопрос: **насколько странно увидеть такие данные,
> если разницы на самом деле нет**. Всё остальное — ваша интерпретация.

---
## Шаг 0. Проект

```
# терминал
mkdir hypothesis-testing
cd hypothesis-testing
```

`requirements.txt`:

```
pandas
scipy
matplotlib
```

```
# терминал (Windows)
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
```

```
# терминал (macOS / Linux)
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

`scipy` — библиотека с реализациями статистических критериев. Все тесты сегодня
берутся из `scipy.stats`.

---
## Данные

1200 рейсов за март–май 2025. Колонки знакомые, добавился `weekday` — день недели.

In [53]:
# ЯЧЕЙКА-ГЕНЕРАТОР ДАННЫХ. Запустите её (Shift+Enter), менять ничего не нужно.
# Создаёт data/trips_stat.csv — 1200 рейсов за март-май 2025.

import csv
import random
from pathlib import Path

random.seed(12)

routes = {
    "Москва-Казань": 820, "Москва-Санкт-Петербург": 710, "Москва-Нижний Новгород": 420,
    "Москва-Воронеж": 520, "Казань-Самара": 360, "Екатеринбург-Челябинск": 220,
    "Новосибирск-Омск": 680, "Санкт-Петербург-Петрозаводск": 430,
}
# у каждого перевозчика своя вероятность задержки и свой уровень тарифа
carriers = {
    "ТрансЛогистик": {"p_delay": 0.28, "price_k": 1.08},
    "РусКарго": {"p_delay": 0.27, "price_k": 1.00},
    "СибПуть": {"p_delay": 0.41, "price_k": 0.99},
    "ВолгаТранс": {"p_delay": 0.18, "price_k": 1.01},
    "АвтоДорСервис": {"p_delay": 0.29, "price_k": 1.00},
}
# нефтепродукты задерживаются чаще остального: оформление, допуски, спецтранспорт
cargo = {
    "Контейнер": {"tariff": 7.5, "delay_k": 1.0},
    "Нефтепродукты": {"tariff": 9.2, "delay_k": 1.7},
    "Зерно": {"tariff": 5.1, "delay_k": 1.0},
    "Металл": {"tariff": 6.8, "delay_k": 1.0},
    "Лес": {"tariff": 4.6, "delay_k": 1.0},
    "Стройматериалы": {"tariff": 5.9, "delay_k": 1.0},
}
weekdays = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"]
days_in_month = {3: 31, 4: 30, 5: 31}

rows = []
trip_no = 1
for month in (3, 4, 5):
    for _ in range(400):
        route, distance = random.choice(list(routes.items()))
        carrier = random.choice(list(carriers))
        cargo_type = random.choice(list(cargo))
        day = random.randint(1, days_in_month[month])

        weight = round(min(22.0, max(2.0, random.gauss(13, 4.5))), 1)

        # вероятность задержки: перевозчик x тип груза (день недели ни на что не влияет)
        p = min(0.9, carriers[carrier]["p_delay"] * cargo[cargo_type]["delay_k"])
        delay = random.choice([1, 1, 2, 2, 3, 4, 6, 8, 12]) if random.random() < p else 0

        speed = random.uniform(46, 64)
        duration = round(distance / speed + delay, 1)
        price = (distance * weight * cargo[cargo_type]["tariff"]
                 * carriers[carrier]["price_k"] * random.uniform(0.93, 1.09))

        rows.append({
            "trip_id": f"T{trip_no:04d}",
            "date": f"2025-{month:02d}-{day:02d}",
            "month": f"2025-{month:02d}",
            "weekday": weekdays[(trip_no + day) % 7],
            "route": route,
            "carrier": carrier,
            "cargo_type": cargo_type,
            "weight_t": weight,
            "distance_km": distance,
            "duration_h": duration,
            "cost_rub": int(round(price, -1)),
            "delay_h": delay,
        })
        trip_no += 1

Path("data").mkdir(exist_ok=True)
with open("data/trips_stat.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print("Готово. Создан файл data/trips_stat.csv")
print("Рейсов:", len(rows))
print("Колонки:", list(rows[0].keys()))

Готово. Создан файл data/trips_stat.csv
Рейсов: 1200
Колонки: ['trip_id', 'date', 'month', 'weekday', 'route', 'carrier', 'cargo_type', 'weight_t', 'distance_km', 'duration_h', 'cost_rub', 'delay_h']


In [54]:
import pandas as pd
from scipy import stats

df = pd.read_csv("data/trips_stat.csv", parse_dates=["date"])
df["is_delayed"] = df["delay_h"] > 0
df["cost_per_km"] = (df["cost_rub"] / df["distance_km"]).round(2)

print("рейсов:", len(df))
print()
print("доля рейсов с задержкой по перевозчикам, %:")
shares = df.groupby("carrier")["is_delayed"].agg(["mean", "count"])
shares["mean"] = (shares["mean"] * 100).round(1)
print(shares.sort_values("mean").to_string())

рейсов: 1200

доля рейсов с задержкой по перевозчикам, %:
               mean  count
carrier                   
ВолгаТранс     22.2    234
РусКарго       27.6    225
ТрансЛогистик  29.7    232
АвтоДорСервис  33.7    255
СибПуть        45.7    254


Разброс большой: от 22.2% до 45.7%. Но прежде чем делать выводы, нужно понять,
какие из этих различий реальны, а какие — случайность выборки.

---
# Часть 1. Нулевая и альтернативная гипотезы

Любая проверка начинается с двух утверждений:

- **H₀ (нулевая гипотеза)** — «разницы нет», «связи нет», «всё одинаково».
  Это утверждение, которое мы пытаемся **опровергнуть**.
- **H₁ (альтернативная)** — «разница есть». То, что мы подозреваем.

**Почему именно так, а не наоборот.** Доказать, что разницы нет, невозможно: отсутствие
доказательств — не доказательство отсутствия. Зато можно показать, что данные плохо
согласуются с «разницы нет», и тогда от этого предположения отказываются.

Логика такая же, как в презумпции невиновности: суд не доказывает невиновность,
он проверяет, достаточно ли улик для обвинения.

**Как формулировать правильно:**

| Вопрос бизнеса | H₀ | H₁ |
|---|---|---|
| У СибПуть больше задержек, чем у ВолгаТранс? | доли задержек равны | доли различаются |
| Нефтепродукты задерживаются чаще? | тип груза и задержка не связаны | связаны |
| ТрансЛогистик берёт дороже? | средняя цена за км одинакова | различается |

**Два правила:**

1. Гипотеза формулируется **до** того, как вы посмотрели на результат. Иначе вы просто
   опишете то, что увидели, и назовёте это открытием.
2. H₀ всегда про **отсутствие** эффекта. «Разница равна нулю», «доли равны», «связи нет».

### Задание 1

Сформулируйте H₀ и H₁ для трёх вопросов. Код не нужен, нужны формулировки.

1. «Зависит ли доля задержек от дня недели?»
2. «Отличается ли средний вес груза у разных перевозчиков?»
3. «Связан ли вес груза со стоимостью рейса?»

> **Мои гипотезы:**
>
> | № | H₀ | H₁ |
> |---|---|---|
> | 1 | Доля задержек не зависит от дня недели | Зависит |
> | 2 | У разных перевозчиков средний вес одинаковый | Различный |
> | 3 | Вес груза не связан со стоимостью рейса | Связан |

---
# Часть 2. Что такое p-value

Самое важное определение пары, и самое часто перевираемое.

> **p-value — это вероятность увидеть такую разницу (или ещё большую),
> если H₀ верна, то есть если разницы на самом деле нет.**

Маленькое p означает: «при отсутствии разницы такие данные были бы редкостью».
Значит, скорее всего, разница есть.

Проще всего понять на монетке. Монета честная (это наша H₀). Подбрасываем 20 раз.

In [55]:
for heads in (10, 12, 15, 18):
    p = stats.binomtest(heads, n=20, p=0.5).pvalue
    verdict = "подозрительно" if p < 0.05 else "обычное дело"
    print(f"{heads:>2} орлов из 20 -> p = {p:.4f}   ({verdict})")

10 орлов из 20 -> p = 1.0000   (обычное дело)
12 орлов из 20 -> p = 0.5034   (обычное дело)
15 орлов из 20 -> p = 0.0414   (подозрительно)
18 орлов из 20 -> p = 0.0004   (подозрительно)


Прочитайте третью строку так: «если монета честная, то 15 и более орлов (или 15 и более
решек) выпадает примерно в 4% случаев». Редко — значит, есть основания усомниться
в честности монеты.

А 12 орлов из 20 — обычное дело: такое и чаще случается с честной монетой,
никаких выводов из этого не сделать.

## Порог 0.05 — это соглашение

Принято считать результат **статистически значимым**, если p < 0.05. Это не закон
природы, а договорённость: мы согласны ошибаться в 5% случаев, объявляя разницу там,
где её нет. В медицине берут 0.01 и строже, в разведочном анализе иногда 0.1.

## Чего p-value НЕ означает — запомните три пункта

| Неверно | Верно |
|---|---|
| «p = 0.03 значит, вероятность что H₀ верна — 3%» | p считается **в предположении**, что H₀ верна; это не вероятность самой H₀ |
| «p = 0.6 доказывает, что разницы нет» | значит лишь, что данных **не хватило**, чтобы её обнаружить |
| «p = 0.001 значит, разница огромная» | p говорит об уверенности, а не о размере. На больших данных значимой становится и ничтожная разница |

Третий пункт — причина, по которой рядом с p всегда смотрят **размер эффекта**:
насколько велика сама разница в понятных единицах.

---
# Часть 3. Сравнение долей: критерий хи-квадрат

Первый практический вопрос: **у СибПуть доля задержек 45.7%, у ВолгаТранс 22.2% —
это реальная разница?**

Данные здесь — две доли (задержался / не задержался). Для сравнения долей строят
**таблицу сопряжённости** 2×2 и применяют критерий хи-квадрат.

```
H₀: доли задержек у двух перевозчиков равны
H₁: доли различаются
```

In [56]:
pair = df[df["carrier"].isin(["СибПуть", "ВолгаТранс"])]
table = pd.crosstab(pair["carrier"], pair["is_delayed"])
print(table)
print()

chi2, p, dof, expected = stats.chi2_contingency(table)

print(f"chi2 = {chi2:.2f}")
print(f"p-value = {p:.6f}")
print()
print("вывод:", "H0 отвергаем — разница значима" if p < 0.05 else "H0 не отвергаем")

is_delayed  False  True 
carrier                 
ВолгаТранс    182     52
СибПуть       138    116

chi2 = 28.63
p-value = 0.000000

вывод: H0 отвергаем — разница значима


p меньше 0.000001 — то есть при одинаковом качестве работы такая разница в 1200 рейсах
практически не могла бы возникнуть случайно. Вывод: СибПуть действительно задерживается чаще.

А теперь сравним двух других перевозчиков, у которых доли близки: 27.6% и 29.7%.

In [57]:
pair2 = df[df["carrier"].isin(["РусКарго", "ТрансЛогистик"])]
table2 = pd.crosstab(pair2["carrier"], pair2["is_delayed"])
print(table2)
print()

chi2_2, p2, _, _ = stats.chi2_contingency(table2)
print(f"chi2 = {chi2_2:.3f}")
print(f"p-value = {p2:.4f}")
print()
print("вывод:", "разница значима" if p2 < 0.05 else "H0 не отвергаем: разница может быть случайной")

is_delayed     False  True 
carrier                    
РусКарго         163     62
ТрансЛогистик    163     69

chi2 = 0.171
p-value = 0.6795

вывод: H0 не отвергаем: разница может быть случайной


**Как правильно сформулировать этот результат.** Неправильно: «разницы между РусКарго
и ТрансЛогистик нет». Правильно: **«данные не дают оснований утверждать, что доли
различаются»**.

Разница в 2.1 процентных пункта вполне может существовать — но на выборке в 225 и 232
рейса мы не можем отличить её от случайных колебаний. Если бы рейсов было в десять раз
больше, тот же разрыв мог бы оказаться значимым.

Это принципиальное различие: **«не нашли» ≠ «нет»**.

### Задание 2

1. Сравните доли задержек у **АвтоДорСервис и ВолгаТранс**. Постройте таблицу,
   посчитайте p, сделайте вывод.
2. Сравните **АвтоДорСервис и ТрансЛогистик**. Какой вывод?
3. В обоих случаях запишите результат двумя формулировками: сухой статистической
   («p = ..., H₀ ...») и на языке бизнеса («мы можем / не можем утверждать, что ...»).

In [58]:
# 1)
pair_a = df[df['carrier'].isin(['АвтоДорСервис', 'ВолгаТранс'])]
table_a = pd.crosstab(pair_a['carrier'], pair_a['is_delayed'])
print(table_a, '\n')
_, p_a, _, _=stats.chi2_contingency(table_a)
print(f'p_value={p_a:.3} H0-отвергаем')
print("вывод: разница значима и не случайна(33,7% и 22,2%)" '\n', '\n')

# 2)
pair_b = df[df['carrier'].isin(['АвтоДорСервис', 'ТрансЛогистик'])]
table_b = pd.crosstab(pair_b['carrier'], pair_b['is_delayed'])
print(table_b, '\n')
_, p_b, _, _=stats.chi2_contingency(table_b)
print(f'p_value={p_b:.3} H0-не отвергаем')
print("вывод: разница может быть случайной(33,7% и 29,7%)")

is_delayed     False  True 
carrier                    
АвтоДорСервис    169     86
ВолгаТранс       182     52 

p_value=0.00647 H0-отвергаем
вывод: разница значима и не случайна(33,7% и 22,2%)
 

is_delayed     False  True 
carrier                    
АвтоДорСервис    169     86
ТрансЛогистик    163     69 

p_value=0.398 H0-не отвергаем
вывод: разница может быть случайной(33,7% и 29,7%)


---
# Часть 4. Сравнение средних: t-тест и Манна-Уитни

Следующий вопрос: **ТрансЛогистик берёт дороже за километр, чем РусКарго?**
Здесь сравниваются не доли, а средние значения непрерывной величины.

| Критерий | Когда применять | Что сравнивает |
|---|---|---|
| **t-тест** (`ttest_ind`) | значения распределены примерно симметрично | средние |
| **Манна-Уитни** (`mannwhitneyu`) | распределение перекошено или есть выбросы | медианы (строго — ранги) |

На практике 10 мы выяснили, что денежные показатели почти всегда перекошены.
Поэтому разумная тактика: **посчитать оба теста**. Если выводы совпали — всё в порядке.
Если разошлись — доверяйте Манна-Уитни и разбирайтесь с выбросами.

Параметр `equal_var=False` в `ttest_ind` стоит ставить почти всегда: он не требует,
чтобы разброс в группах был одинаковым.

In [59]:
x = df[df["carrier"] == "ТрансЛогистик"]["cost_per_km"]
y = df[df["carrier"] == "РусКарго"]["cost_per_km"]

print(f"ТрансЛогистик: среднее {x.mean():.2f}, медиана {x.median():.2f}, n = {len(x)}")
print(f"РусКарго:      среднее {y.mean():.2f}, медиана {y.median():.2f}, n = {len(y)}")
print()

t_stat, p_t = stats.ttest_ind(x, y, equal_var=False)
print(f"t-тест:        t = {t_stat:.3f},  p = {p_t:.5f}")

u_stat, p_u = stats.mannwhitneyu(x, y, alternative="two-sided")
print(f"Манна-Уитни:   U = {u_stat:.0f}, p = {p_u:.5f}")
print()
print("оба теста:", "значимо" if p_t < 0.05 and p_u < 0.05 else "выводы расходятся")

ТрансЛогистик: среднее 90.28, медиана 87.75, n = 232
РусКарго:      среднее 81.55, медиана 76.38, n = 225

t-тест:        t = 2.615,  p = 0.00921
Манна-Уитни:   U = 29994, p = 0.00580

оба теста: значимо


## Размер эффекта: значимо — не значит важно

p сказал, что разница реальна. Но насколько она велика? Для этого считают **размер
эффекта**. Самая простая мера для средних — **d Коэна**: разница средних, выраженная
в стандартных отклонениях.

| d | Интерпретация |
|---|---|
| около 0.2 | маленький эффект |
| около 0.5 | средний |
| около 0.8 и выше | большой |

In [60]:
n1, n2 = len(x), len(y)
pooled_sd = (((n1 - 1) * x.std() ** 2 + (n2 - 1) * y.std() ** 2) / (n1 + n2 - 2)) ** 0.5
cohen_d = (x.mean() - y.mean()) / pooled_sd

print(f"разница средних: {x.mean() - y.mean():.2f} руб./км "
      f"({(x.mean() / y.mean() - 1) * 100:.1f}%)")
print(f"d Коэна: {cohen_d:.2f}")
print()
print("статистически значимо, но эффект по шкале Коэна небольшой")

разница средних: 8.73 руб./км (10.7%)
d Коэна: 0.24

статистически значимо, но эффект по шкале Коэна небольшой


Вот как это звучит в отчёте: *«ТрансЛогистик дороже в среднем на 8.7 руб./км (+10.7%),
различие статистически значимо (p = 0.009), размер эффекта небольшой (d = 0.24) —
то есть тарифы групп сильно перекрываются»*.

Последняя часть важна: небольшой d означает, что много рейсов РусКарго дороже,
чем у ТрансЛогистик, несмотря на разницу в средних. Вывод «ТрансЛогистик дороже»
верен для среднего, но не для каждого конкретного рейса.

### Задание 3

Мы уже знаем, что у СибПуть задержек вдвое больше, чем у ВолгаТранс (45.7% против 22.2%).
Проверим, видно ли это в других показателях.

1. Сравните **среднее время в пути** (`duration_h`) у СибПуть и ВолгаТранс.
   Примените оба критерия и посчитайте d Коэна. Значима ли разница?
2. Сравните **саму задержку** (`delay_h`) у этих же перевозчиков. А здесь значима?
3. Сравните **средний вес груза** (`weight_t`). Ожидали ли вы тут разницу?
4. **Главный вопрос.** По задержке разница есть, а по общему времени в пути —
   не обнаружена, хотя задержка входит в это время. Как такое возможно?

   Чтобы ответить, посмотрите на два числа:
   ```python
   print(df["duration_h"].std(), df["delay_h"].std())
   print(df.groupby("route")["duration_h"].mean())
   ```
5. Предложите показатель, в котором эффект будет видно. Подсказка: нужно убрать
   влияние расстояния — например, посчитать время в пути **на 100 км**
   (`duration_h / distance_km * 100`) и сравнить его.

In [61]:
# 1) время в пути
print('№1')
a = df[df['carrier']=='СибПуть']['duration_h']
b = df[df['carrier']=='ВолгаТранс']['duration_h']
t_stat, p_t = stats.ttest_ind(a, b, equal_var=False)
print(f"t-тест:        t = {t_stat:.3f},  p = {p_t:.5f}")
u_stat, p_u = stats.mannwhitneyu(a, b, alternative="two-sided")
print(f"Манна-Уитни:   U = {u_stat:.0f}, p = {p_u:.5f}")
n_a, n_b = len(a), len(b)
pooled_sd = (((n_a - 1) * a.std() ** 2 + (n_b - 1) * b.std() ** 2) / (n_a + n_b - 2)) ** 0.5
cohen_d = (a.mean() - b.mean()) / pooled_sd
print(f"d Коэна: {cohen_d:.2f}")
print('Разница не значима(p>0.05 в обоих критериях)', '\n')

# 2) задержка
print('№2')
c = df[df['carrier']=='СибПуть']['delay_h']
d = df[df['carrier']=='ВолгаТранс']['delay_h']
t_stat, p_t = stats.ttest_ind(c, d, equal_var=False)
print(f"t-тест:        t = {t_stat:.3f},  p = {p_t:.5f}")
u_stat, p_u = stats.mannwhitneyu(c, d, alternative="two-sided")
print(f"Манна-Уитни:   U = {u_stat:.0f}, p = {p_u:.5f}")
n_c, n_d = len(c), len(d)
pooled_sd = (((n_c - 1) * c.std() ** 2 + (n_d - 1) * d.std() ** 2) / (n_c + n_d - 2)) ** 0.5
cohen_d = (c.mean() - d.mean()) / pooled_sd
print(f"d Коэна: {cohen_d:.2f}")
print('Разница значима(p<0.05 в обоих критериях), эффект по шкале Коэна небольшой', '\n')

# 3) вес груза
print('№3')
e = df[df['carrier']=='СибПуть']['weight_t']
f = df[df['carrier']=='ВолгаТранс']['weight_t']
t_stat, p_t = stats.ttest_ind(e, f, equal_var=False)
print(f"t-тест:        t = {t_stat:.3f},  p = {p_t:.5f}")
u_stat, p_u = stats.mannwhitneyu(e, f, alternative="two-sided")
print(f"Манна-Уитни:   U = {u_stat:.0f}, p = {p_u:.5f}")
n_e, n_f = len(e), len(f)
pooled_sd = (((n_e - 1) * e.std() ** 2 + (n_f - 1) * f.std() ** 2) / (n_e + n_f - 2)) ** 0.5
cohen_d = (e.mean() - f.mean()) / pooled_sd
print(f"d Коэна: {cohen_d:.2f}")
print('Разница не значима(p>0.05 в обоих критериях)', '\n')

# 4)
print('№4')
print("""Из-за того, что std общей длительности рейса больше чем std задержек(4,6 против 2,9), т.к. среднее значение 
продолжительности рейсов сильно различается по маршрутам (от 5,7 до 16,3), разница в задержках 
теряется на фоне такого большого разброса общей продолжительности рейса""", '\n')

# 5) время на 100 км
print('№5')
new=df.copy()
new['duration_100km']=new['duration_h']/df['distance_km']*100
g = new[new['carrier']=='СибПуть']['duration_100km']
h = new[new['carrier']=='ВолгаТранс']['duration_100km']
t_stat, p_t = stats.ttest_ind(g, h, equal_var=False)
print(f"t-тест:        t = {t_stat:.3f},  p = {p_t:.5f}")
u_stat, p_u = stats.mannwhitneyu(g, h, alternative="two-sided")
print(f"Манна-Уитни:   U = {u_stat:.0f}, p = {p_u:.5f}")
n_g, n_h = len(g), len(h)
pooled_sd = (((n_g - 1) * g.std() ** 2 + (n_h - 1) * h.std() ** 2) / (n_g + n_h - 2)) ** 0.5
cohen_d = (g.mean() - h.mean()) / pooled_sd
print(f"d Коэна: {cohen_d:.2f}")
print('Разница не значима(p<0.05 в обоих критериях), эффект по шкале Коэна небольшой')
   

№1
t-тест:        t = 1.351,  p = 0.17722
Манна-Уитни:   U = 30906, p = 0.44522
d Коэна: 0.12
Разница не значима(p>0.05 в обоих критериях) 

№2
t-тест:        t = 3.936,  p = 0.00010
Манна-Уитни:   U = 36576, p = 0.00000
d Коэна: 0.35
Разница значима(p<0.05 в обоих критериях), эффект по шкале Коэна небольшой 

№3
t-тест:        t = -0.340,  p = 0.73419
Манна-Уитни:   U = 29218, p = 0.74823
d Коэна: -0.03
Разница не значима(p>0.05 в обоих критериях) 

№4
Из-за того, что std общей длительности рейса больше чем std задержек(4,6 против 2,9), т.к. среднее значение 
продолжительности рейсов сильно различается по маршрутам (от 5,7 до 16,3), разница в задержках 
теряется на фоне такого большого разброса общей продолжительности рейса 

№5
t-тест:        t = 2.536,  p = 0.01154
Манна-Уитни:   U = 34874, p = 0.00092
d Коэна: 0.23
Разница не значима(p<0.05 в обоих критериях), эффект по шкале Коэна небольшой


---
# Часть 5. Связь двух категорий

Вопрос: **зависит ли задержка от типа груза?** Здесь обе переменные категориальные,
и групп больше двух. Тот же критерий хи-квадрат, только таблица больше.

```
H₀: тип груза и наличие задержки независимы
H₁: связь есть
```

In [62]:
ct = pd.crosstab(df["cargo_type"], df["is_delayed"])
ct_view = ct.copy()
ct_view["доля задержек, %"] = (ct[True] / (ct[True] + ct[False]) * 100).round(1)
print(ct_view.sort_values("доля задержек, %").to_string())
print()

chi2_c, p_c, dof_c, _ = stats.chi2_contingency(ct)
print(f"chi2 = {chi2_c:.2f}, p = {p_c:.8f}, степеней свободы = {dof_c}")
print("вывод:", "связь есть" if p_c < 0.05 else "связи не обнаружено")

is_delayed      False  True  доля задержек, %
cargo_type                                   
Металл            149    53              26.2
Контейнер         126    45              26.3
Лес               163    60              26.9
Зерно             154    72              31.9
Стройматериалы    122    62              33.7
Нефтепродукты     101    93              47.9

chi2 = 31.13, p = 0.00000884, степеней свободы = 5
вывод: связь есть


Связь значима, и по таблице видно, за счёт чего: нефтепродукты задерживаются
почти в два раза чаще остальных грузов.

> **Важно:** хи-квадрат отвечает только «связь есть или нет». Он **не говорит**,
> какая именно группа выбивается — это вы смотрите глазами по таблице долей.

Теперь контрольная проверка: день недели. Никаких оснований ожидать здесь связь нет —
посмотрим, что скажет критерий.

In [63]:
ct_wd = pd.crosstab(df["weekday"], df["is_delayed"])
print((ct_wd[True] / (ct_wd[True] + ct_wd[False]) * 100).round(1).to_string())
print()

chi2_w, p_w, dof_w, _ = stats.chi2_contingency(ct_wd)
print(f"chi2 = {chi2_w:.2f}, p = {p_w:.4f}")
print("вывод:", "связь есть" if p_w < 0.05 else "связи не обнаружено")

weekday
Вс    33.9
Вт    36.1
Пн    32.4
Пт    32.1
Сб    28.1
Ср    32.2
Чт    30.3

chi2 = 2.98, p = 0.8111
вывод: связи не обнаружено


Доли по дням недели колеблются от 28% до 36% — на глаз кажется, что суббота лучше
понедельника. Критерий показывает: p = 0.81, это обычный случайный разброс.

**Это и есть главная польза статистики в работе аналитика.** Глаз всегда найдёт
закономерность — даже там, где её нет. Критерий защищает от выводов на пустом месте.

### Задание 4

1. Проверьте связь между **маршрутом** и наличием задержки. Есть ли она?
2. Проверьте связь между **месяцем** и наличием задержки.
3. Для того теста, где связь нашлась, посмотрите таблицу долей и назовите группу,
   которая выбивается сильнее всего.
4. Что означает результат для теста, где связь не нашлась? Запишите формулировку.

In [64]:
# 1)
ct_r=pd.crosstab(df['route'], df['is_delayed'])
_, p_r, _, _ = stats.chi2_contingency(ct_r)
print(f"p = {p_r:.4f}")
print("вывод:", "связь есть" if p_r < 0.05 else "связи не обнаружено")
print('Н0 не отвергаем, связи между маршрутом и наличием задержки нет', '\n')

# 2)
ct_m=pd.crosstab(df['month'], df['is_delayed'])
_, p_m, _, _ = stats.chi2_contingency(ct_m)
print(f"p = {p_m:.4f}")
print("вывод:", "связь есть" if p_m < 0.05 else "связи не обнаружено")
print('Н0 не отвергаем, связи между месяцем и наличием задержки нет')

p = 0.5141
вывод: связи не обнаружено
Н0 не отвергаем, связи между маршрутом и наличием задержки нет 

p = 0.8484
вывод: связи не обнаружено
Н0 не отвергаем, связи между месяцем и наличием задержки нет


---
# Часть 6. Две ловушки

## Ловушка 1: p зависит от размера выборки

Одна и та же разница на маленькой выборке незначима, на большой — значима.
В примере ниже разница между группами **всегда одинакова** (3 единицы), меняется
только число наблюдений.

In [65]:
import numpy as np

rng = np.random.default_rng(7)

print("Разница между группами всегда одна и та же (средние 100 и 103):")
for n in (20, 50, 100, 300, 1000):
    a = rng.normal(100, 15, n)
    b = rng.normal(103, 15, n)
    _, p_n = stats.ttest_ind(a, b)
    mark = "<- значимо" if p_n < 0.05 else ""
    print(f"  n = {n:>4} в каждой группе:  p = {p_n:.4f}  {mark}")

Разница между группами всегда одна и та же (средние 100 и 103):
  n =   20 в каждой группе:  p = 0.8720  
  n =   50 в каждой группе:  p = 0.9848  
  n =  100 в каждой группе:  p = 0.0274  <- значимо
  n =  300 в каждой группе:  p = 0.0000  <- значимо
  n = 1000 в каждой группе:  p = 0.0000  <- значимо


Отсюда два практических следствия:

- **p > 0.05 при маленькой выборке** ничего не доказывает: возможно, просто мало данных;
- **p < 0.05 при огромной выборке** не означает важности: на миллионе наблюдений значимой
  станет разница в сотые доли процента. Поэтому всегда смотрите размер эффекта.

## Ловушка 2: множественные сравнения

Если проверять много гипотез подряд, часть «значимых» результатов окажется случайной.
При пороге 0.05 примерно **каждый двадцатый** тест даёт ложное срабатывание.

Проверим на заведомо одинаковых группах — сравним случайные числа сами с собой 20 раз.

In [66]:
rng = np.random.default_rng(42)
false_positives = 0

for i in range(20):
    g1 = rng.normal(100, 15, 200)
    g2 = rng.normal(100, 15, 200)          # ровно то же распределение!
    _, p_i = stats.ttest_ind(g1, g2)
    if p_i < 0.05:
        false_positives += 1
        print(f"  сравнение {i + 1:>2}: p = {p_i:.4f}  <- ложная находка")

print()
print(f"групп с реальной разницей: 0, а 'значимых' результатов: {false_positives} из 20")

  сравнение  9: p = 0.0400  <- ложная находка

групп с реальной разницей: 0, а 'значимых' результатов: 1 из 20


Группы были одинаковыми по построению — и всё равно нашлось «открытие».

**Что с этим делать.** Самый простой способ — **поправка Бонферрони**: порог делится
на число сравнений. Проверяете 10 пар — используйте порог 0.05 / 10 = 0.005.

Способ надёжнее и проще: **решить заранее**, какие именно гипотезы вы проверяете,
и не перебирать все комбинации в поисках красивого p.

In [67]:
# Все пары перевозчиков: 10 сравнений
carriers = sorted(df["carrier"].unique())
rows = []

for i in range(len(carriers)):
    for j in range(i + 1, len(carriers)):
        sub = df[df["carrier"].isin([carriers[i], carriers[j]])]
        tbl = pd.crosstab(sub["carrier"], sub["is_delayed"])
        _, p_pair, _, _ = stats.chi2_contingency(tbl)
        rows.append({"A": carriers[i], "B": carriers[j], "p": round(p_pair, 4)})

pairs = pd.DataFrame(rows).sort_values("p")
pairs["p < 0.05"] = pairs["p"] < 0.05
pairs["p < 0.005 (Бонферрони)"] = pairs["p"] < 0.05 / len(pairs)

print(pairs.to_string(index=False))
print()
print("значимых при обычном пороге:", pairs["p < 0.05"].sum())
print("значимых после поправки:    ", pairs["p < 0.005 (Бонферрони)"].sum())

            A             B      p  p < 0.05  p < 0.005 (Бонферрони)
   ВолгаТранс       СибПуть 0.0000      True                    True
     РусКарго       СибПуть 0.0001      True                    True
      СибПуть ТрансЛогистик 0.0004      True                    True
АвтоДорСервис    ВолгаТранс 0.0065      True                   False
АвтоДорСервис       СибПуть 0.0077      True                   False
   ВолгаТранс ТрансЛогистик 0.0809     False                   False
АвтоДорСервис      РусКарго 0.1733     False                   False
   ВолгаТранс      РусКарго 0.2248     False                   False
АвтоДорСервис ТрансЛогистик 0.3979     False                   False
     РусКарго ТрансЛогистик 0.6795     False                   False

значимых при обычном пороге: 5
значимых после поправки:     3


### Задание 5

1. Сколько сравнений «не прошли» поправку Бонферрони? Назовите эти пары.
2. Представьте, что вы проверили только одну пару — ту, где p = 0.0065 — и нашли её
   «значимой». Насколько надёжен такой вывод, если вы перебрали все 10 пар, прежде
   чем выбрать эту? Объясните в двух предложениях.
3. Как нужно было поступить правильно? Сформулируйте, что следовало сделать **до**
   расчётов.

> **Ответы:**
>
> 1. 2 сравнения: АвтоДорСервис-ВолгаТранс и АвтоДорСервис-СибПуть
> 2. Вывод ненадёжен: если проверять много гипотез подряд, часть «значимых» результатов окажется случайной, чтобы убедиться что она действительно "значима" нужно использовать поправку Бонферрони-порог разделить на число сравнений. В данном случае она не будет значимой (0.0065>0.05/10)
> 3. Следовало решить заранее, какие именно гипотезы проверяются, и не перебирать все комбинации в поисках красивого p.

---
# Часть 7. Отчёт о проверке гипотез

Результат проверки нельзя подавать как «p = 0.0092». Руководитель не знает, что это,
и знать не обязан. Отчёт строится по схеме:

| Блок | Что пишем |
|---|---|
| Вопрос | на человеческом языке |
| Что сравнивали | группы и показатель, с размерами выборок |
| Результат | числа: доли или средние |
| Вывод | есть разница или данных не хватило |
| Размер эффекта | насколько велика разница в понятных единицах |
| Оговорки | чего этот результат **не** доказывает |

Техническая часть (название критерия, статистика, p) идёт в приложение или сноску —
её читают те, кто будет перепроверять.

### Задание 6

Соберите итоговую таблицу по всем проверкам пары и напишите текст для руководителя.

In [68]:
# Сводная таблица результатов: заполните по итогам заданий
results = pd.DataFrame([
    {"гипотеза": "СибПуть vs ВолгаТранс: доли задержек",
     "критерий": "хи-квадрат", "p": '<0.0001', "вывод": "разница значима"},
    {"гипотеза": "РусКарго vs ТрансЛогистик: доли задержек",
     "критерий": "хи-квадрат", "p": 0.6795, "вывод": "разница случайна"},
    {"гипотеза": "ТрансЛогистик vs РусКарго: цена за км",
     "критерий": "t-тест / Манна-Уитни", "p": '0.0092 / 0.0058', "вывод": "статистически значимо, но эффект по шкале Коэна небольшой"},
    {"гипотеза": "тип груза и задержки",
     "критерий": "хи-квадрат", "p": '<0.0001', "вывод": "связь есть"},
    {"гипотеза": "день недели и задержки",
     "критерий": "хи-квадрат", "p": 0.811, "вывод": "связи не обнаружено"},
])

# подставьте посчитанные p и выводы
results

,гипотеза,критерий,p,вывод
0,СибПуть vs ВолгаТранс: доли задержек,хи-квадрат,<0.0001,разница значима
1,РусКарго vs ТрансЛогистик: доли задержек,хи-квадрат,0.6795,разница случайна
2,ТрансЛогистик vs РусКарго: цена за км,t-тест / Манна-Уитни,0.0092 / 0.0058,"статистически значимо, но эффект по шкале Коэн..."
3,тип груза и задержки,хи-квадрат,<0.0001,связь есть
4,день недели и задержки,хи-квадрат,0.811,связи не обнаружено


### Текст для руководителя

Напишите 4–5 предложений по результатам всех проверок. Требования:

- без слов «p-value», «хи-квадрат», «нулевая гипотеза»;
- каждое утверждение с числом;
- отдельно сказано, что **не** удалось установить;
- в конце — что делать дальше.

> **Отчёт:**
>
>  Чаще всего срывает сроки СибПуть (задержки в 45,7% случаев против 22,2% у ВолгаТранса), а среди категорий товаров наиболее проблемными оказались нефтепродукты (48% опозданий против 26-34% у прочих грузов). ТрансЛогистик дороже РусКарго в среднем на 8.7 руб./км (+10.7%), но тарифы групп сильно перекрываются. При этом подтвердить разницу в пунктуальности между РусКарго (27,6% задержек) и ТрансЛогистиком (29,7%), а также зависимость опозданий от дня недели не удалось - колебания в пределах случайности. Рекомендую провести встречу с СибПутём для разбора причин сбоев и закладывать дополнительный буфер времени при планировании рейсов с нефтепродуктами.

---
## Шпаргалка

```python
from scipy import stats

# Сравнить две доли (или связь двух категорий)
table = pd.crosstab(df["группа"], df["признак"])
chi2, p, dof, expected = stats.chi2_contingency(table)

# Сравнить средние двух групп
t, p = stats.ttest_ind(x, y, equal_var=False)      # симметричные данные
u, p = stats.mannwhitneyu(x, y, alternative="two-sided")   # перекошенные

# Проверить корреляцию
r, p = stats.pearsonr(df["a"], df["b"])            # линейная связь
rho, p = stats.spearmanr(df["a"], df["b"])         # монотонная связь

# Размер эффекта для средних
pooled = (((n1-1)*x.std()**2 + (n2-1)*y.std()**2) / (n1+n2-2)) ** 0.5
d = (x.mean() - y.mean()) / pooled
```

**Выбор критерия:**

| Что сравниваем | Критерий |
|---|---|
| две доли | хи-квадрат на таблице 2×2 |
| две категориальные переменные | хи-квадрат на таблице сопряжённости |
| средние двух групп, данные симметричны | t-тест |
| средние двух групп, данные перекошены | Манна-Уитни |
| связь двух числовых величин | корреляция Пирсона или Спирмена |

## Отправляем работу

```
# терминал
git add .
git commit -m "Практика 12: проверка статистических гипотез"
git push
```

## Чек-лист сдачи

- [ ] Задание 1: гипотезы сформулированы, H₀ везде про отсутствие эффекта
- [ ] Задание 2: два сравнения долей, выводы в двух формулировках
- [ ] Задание 3: t-тест и Манна-Уитни, посчитан размер эффекта
- [ ] Задание 4: два теста на связь категорий, объяснён отрицательный результат
- [ ] Задание 5: разобрана проблема множественных сравнений
- [ ] Задание 6: сводная таблица и текст для руководителя без жаргона
- [ ] Всё запушено в репозиторий

## Что унести с пары

1. **p-value — это «насколько странны данные, если разницы нет»**, а не вероятность
   того, что вы правы.
2. **«Не нашли разницу» ≠ «разницы нет».** Возможно, просто не хватило данных.
3. **Значимо ≠ важно.** Всегда смотрите размер эффекта рядом с p.
4. **Гипотеза формулируется до расчёта.** Перебрать двадцать вариантов и выбрать тот,
   где p получилось маленьким, — это не анализ, это самообман.

И главное, ради чего тема существует: глаз всегда видит закономерность. Разница
в задержках по дням недели выглядела убедительно — а её не было. Статистический
критерий нужен именно для того, чтобы отличать настоящие находки от случайного шума.